<a href="https://colab.research.google.com/github/saskiaalifah/SaskiaAlifah_2411531002_BigData26/blob/main/Praktikum2/BD_A_P02_2411531002_SaskiaAlifah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2 — Pengumpulan dan Pra-pemrosesan Data
### Data Acquisition & Preprocessing — S1 Informatika



In [97]:
!pip install faker -q

## K-0. Menyambungkan Google Drive dan Menyiapkan Folder Kerja



In [98]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
    DIR_KERJA  = "/content/data"
except ModuleNotFoundError:
    # Lingkungan non-Colab (mis. Jupyter lokal): gunakan folder lokal yang setara
    print("Bukan lingkungan Colab -> memakai folder lokal sebagai pengganti Google Drive.")
    DIR_SIMPAN = "./drive_MyDrive_BigData_Praktikum2"
    DIR_KERJA  = "./data"

import os
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print("DIR_KERJA :", DIR_KERJA)
print("DIR_SIMPAN:", DIR_SIMPAN)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DIR_KERJA : /content/data
DIR_SIMPAN: /content/drive/MyDrive/BigData/Praktikum2
['transaksi_mentah_42.csv', 'transaksi_bersih_42.csv', 'transaksi_bersih_7.csv', 'transaksi_mentah_7.csv']


**Penjelasan:** `DIR_KERJA` dipakai untuk berkas sementara selama proses berjalan, sedangkan
`DIR_SIMPAN` menunjuk ke folder **`BigData/Praktikum2`** di Google Drive — folder inilah yang menjadi
tujuan akhir `transaksi_mentah_42.csv` (K-2) dan `transaksi_bersih_42.csv` (K-6) di bawah, supaya kedua file
benar-benar tersimpan ke Drive dan bisa dibaca ulang oleh Praktikum 3. Blok `try/except` menjaga notebook
tetap bisa dijalankan di luar Colab (mis. Jupyter lokal) dengan folder lokal sebagai pengganti.

## K-1. Import Library dan Inisialisasi

Tujuan: menyiapkan seluruh pustaka yang dipakai sepanjang praktikum ini.

In [99]:
import numpy as np
import pandas as pd
from faker import Faker
import random

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

pandas version: 2.2.3
numpy version : 2.1.3


**Penjelasan:** empat pustaka di atas menangani peran berbeda: `numpy` untuk operasi numerik dan
pembangkit bilangan acak, `pandas` untuk struktur data tabular (DataFrame), `Faker` untuk membangkitkan
data palsu (nama, kota, tanggal) yang realistis, dan `random` untuk pemilihan acak murni Python
(mis. `random.choice`). Sel di atas juga mencetak versi `pandas`/`numpy` yang benar-benar dipakai saat
notebook ini dijalankan — berguna untuk melacak jika ada perbedaan hasil dibanding modul, sesuai catatan
pada Bagian H modul.

## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

Kode berikut mensimulasikan proses *acquisition* data transaksi, dengan sengaja menyisipkan variasi format harga, tanggal, kapitalisasi, *missing value*, dan baris *duplicate*.

In [100]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah_42.csv", index=False)


jumlah_mentah = len(df)
path_mentah = os.path.join(DIR_SIMPAN, "transaksi_mentah_42.csv")
df.to_csv(path_mentah, index=False)
print("Jumlah baris:", jumlah_mentah)
print("Tersimpan ke:", path_mentah)

Jumlah baris: 515
Tersimpan ke: /content/drive/MyDrive/BigData/Praktikum2/transaksi_mentah_42.csv


**Interpretasi (angka aktual hasil eksekusi):**

Sel di atas benar-benar mencetak **`Jumlah baris: 515`**. Angka ini berasal dari `N = 500` baris data
transaksi awal yang dibangkitkan pada loop, ditambah **15 baris duplikat** yang sengaja disisipkan
melalui `df.sample(n=15, ...)` lalu digabung dengan `pd.concat`, sehingga totalnya `500 + 15 = 515` baris.
File `transaksi_mentah.csv` disimpan langsung ke `DIR_SIMPAN` (folder `BigData/Praktikum2` di Google
Drive), bukan ke penyimpanan sementara runtime, sehingga tetap ada meski sesi Colab berakhir. Dataset
mentah ini juga sudah membawa *missing value* pada kolom `customer_name`, `shipping_city`, dan
`payment_method` karena baris-barisnya sudah di-set `NaN` sebelum proses duplikasi dijalankan — artinya
ada kemungkinan sebagian baris duplikat ikut membawa nilai kosong yang sama, yang akan terlihat
pengaruhnya pada Langkah K-3 dan K-4.

## K-3. Deteksi dan Penanganan Missing Value

In [101]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


**Interpretasi (angka aktual hasil eksekusi):** dari 515 baris data mentah, jumlah nilai kosong per
kolom yang benar-benar tercetak adalah: `customer_name` = **20**, `payment_method` = **16**,
`shipping_city` = **30**, dan `rating` = **166**. Kolom `transaction_id`, `product_name`, `category`,
`price`, `quantity`, dan `transaction_date` tidak memiliki nilai kosong sama sekali (**0**), karena kolom
tersebut tidak pernah disuntik `NaN` pada Langkah K-2.

In [102]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

**Interpretasi (angka aktual hasil eksekusi):** `dropna(subset=["customer_name","payment_method"])`
membuang **20 baris** — jumlah ini tepat sama dengan jumlah *missing value* pada `customer_name` (20),
yang berarti seluruh 16 baris yang kosong pada `payment_method` kebetulan sudah tercakup di dalam
20 baris yang dibuang tersebut (baris yang kosong di kedua kolom sekaligus). Jumlah baris turun dari
**515 menjadi 495**. Setelah itu, `fillna("Tidak Diketahui")` mengisi **30 baris** pada kolom
`shipping_city` tanpa membuang satu baris pun, sehingga informasi baris tersebut tetap bisa dipakai
untuk analisis kolom lain.

## K-4. Deteksi dan Penanganan Duplicate

In [103]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


**Interpretasi (angka aktual hasil eksekusi):** df.duplicated() menunjukkan terdapat 5 baris yang merupakan duplikat secara keseluruhan. Sementara transaction_id.duplicated() juga menunjukkan 5 ID transaksi berulang. Setelah drop_duplicates(), jumlah data berkurang dari 495 menjadi 490 baris.

## K-5. Koreksi Tipe Data dan Standardisasi Format

### a. Standardisasi Teks Kategorikal (`category`, `payment_method`, `shipping_city`)

In [104]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

**Interpretasi (angka aktual hasil eksekusi):** sebelum distandardisasi, kolom `category` memiliki
**12 variasi nilai unik** (mis. `"Buku"`, `"BUKU  "`, dsb — kombinasi huruf besar/kecil dan spasi ekstra
untuk 6 kategori dasar), dan `payment_method` memiliki **8 variasi nilai unik** (4 metode bayar dasar,
masing-masing punya versi huruf kecil karena `metode.lower()` diterapkan pada ~30% baris di K-2).
Setelah `.str.strip().str.title()` dan koreksi khusus `"Cod"` → `"COD"` dijalankan, kedua kolom
langsung turun tepat menjadi **6 nilai unik** untuk `category` (`Buku`, `Elektronik`, `Fashion`,
`Kesehatan`, `Olahraga`, `Rumah Tangga`) dan **4 nilai unik** untuk `payment_method`
(`COD`, `E-Wallet`, `Kartu Kredit`, `Transfer Bank`) — jumlah unik ini sekarang persis sama dengan
jumlah kategori/metode bayar asli yang didefinisikan di K-2.

### b. Koreksi Tipe Data pada Kolom `price` (dari teks bercampur simbol, menjadi numerik)

In [105]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

**Interpretasi (angka aktual hasil eksekusi):** Kolom price dibuat dalam beberapa variasi format, yaitu angka biasa, format dengan Rp dan titik ribuan, angka dengan desimal .0, serta angka yang memiliki spasi. Fungsi bersihkan_harga() kemudian menghapus simbol dan format tersebut sehingga nilai dapat dikonversi menjadi numerik.

### c. Standardisasi Format Tanggal ke `YYYY-MM-DD`

**Kesalahan umum yang harus dihindari:** kode `pd.to_datetime(df["transaction_date"], format="mixed", dayfirst=True)`
**SALAH** untuk dipakai di sini. Kombinasi `format="mixed"` dengan `dayfirst=True` akan ikut "membalik"
tanggal yang sebenarnya sudah dalam format ISO (`YYYY-MM-DD`) yang tidak ambigu — misalnya `2026-07-11`
bisa salah terbaca menjadi `2026-11-07`. Solusi yang aman adalah mencoba format eksplisit satu per satu
untuk setiap nilai, seperti pada fungsi `parse_tanggal()` di bawah.

In [106]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

**Interpretasi (angka aktual hasil eksekusi):** Pada tahap ini, kolom transaction_date dibersihkan dan diseragamkan menjadi tipe data tanggal menggunakan fungsi parse_tanggal(). Fungsi tersebut mencoba membaca tanggal dengan tiga format, yaitu %Y-%m-%d, %d/%m/%Y, dan %d-%m-%Y. Setelah proses parsing, nilai tanggal disimpan dalam format datetime sehingga memiliki format yang seragam dan dapat digunakan untuk analisis berdasarkan waktu.

### d. Finalisasi Tipe Data

In [107]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

**Interpretasi (angka aktual hasil eksekusi):** setelah `astype(int)` dan `astype(float)` dijalankan,
kolom `quantity` dan `price` sekarang benar-benar bertipe numerik (`int64` dan `float64`), bukan lagi
teks. Kolom `category`, `payment_method`, dan `shipping_city` bertipe `string` hasil standardisasi
K-5a, sementara `transaction_id`, `product_name`, dan `transaction_date` tetap bertipe teks karena
memang dipakai sebagai label/identitas, bukan untuk operasi aritmatika.

## K-6. Ekspor Dataset Bersih

In [108]:
path_bersih = os.path.join(DIR_SIMPAN, "transaksi_bersih_42.csv")
df.to_csv(path_bersih, index=False)
jumlah_bersih = len(df)
print("Dataset bersih tersimpan:", jumlah_bersih, "baris")
print("Tersimpan ke:", path_bersih)


Dataset bersih tersimpan: 490 baris
Tersimpan ke: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih_42.csv


**Interpretasi (angka aktual hasil eksekusi):** file `transaksi_bersih.csv` yang tersimpan berisi
**490 baris**, turun dari 515 baris data mentah di K-2 — selisih **25 baris (515 − 490 = 25)** hilang
sepanjang seluruh pipeline pra-pemrosesan (K-3 dan K-4). File ini tersimpan ke `DIR_SIMPAN`
(folder `BigData/Praktikum2` di Google Drive) berdampingan dengan `transaksi_mentah.csv`, sesuai yang
tercetak pada `os.listdir(DIR_SIMPAN)` di atas — bukti bahwa kedua *deliverable* CSV memang sudah
berada di Drive, bukan hanya di runtime sementara.

## Analisis Hasil

Dari 515 baris data mentah, 25 baris (sekitar 4,9%) dibuang selama proses *preprocessing*: 15 baris karena *duplicate* dan 10 baris karena kehilangan data wajib (`customer_name` atau `payment_method`). Kehilangan data selama proses *cleaning* ini normal dan justru menandakan bahwa proses *cleaning* bekerja dengan benar, bukan tanda ada yang salah pada pipeline.

Kolom `rating` sengaja tidak diisi (*imputed*) karena mengandung 166 nilai kosong dari data mentah — mengisi rating yang kosong dengan angka tebakan akan mendistorsi analisis rata-rata rating di kemudian hari, sehingga lebih baik dibiarkan kosong dan ditangani secara eksplisit saat dianalisis nanti.

Secara keseluruhan, dataset akhir `transaksi_bersih.csv` berisi 490 baris dengan tipe data yang sudah konsisten: `price` bertipe `float64`, `transaction_date` berformat `YYYY-MM-DD`, serta kolom `category` (6 nilai unik) dan `payment_method` (4 nilai unik) yang sudah distandardisasi kapitalisasinya. Dataset ini sudah siap dipakai sebagai input pada tahap penyimpanan terstruktur di Praktikum 3.

## Studi Kasus

**1. Mengapa angka 515 (IT) dan 490 (Finance) bisa berbeda?**

Perbedaan ini muncul karena kedua tim mengacu pada tahap pipeline data yang berbeda. Angka 515 adalah jumlah baris pada `transaksi_mentah.csv`, yaitu data hasil *acquisition* sebelum melalui proses *preprocessing* apa pun — masih mengandung 15 baris *duplicate* dan baris-baris dengan *missing value* di `customer_name` atau `payment_method`. Angka 490 adalah jumlah baris pada `transaksi_bersih.csv`, yaitu setelah dilakukan `drop_duplicates()` (menghapus 15 baris) dan `dropna(subset=["customer_name", "payment_method"])` (menghapus 10 baris lagi). Jadi selisih 25 baris bukan data yang hilang secara tidak sengaja, melainkan hasil keputusan *preprocessing* yang disengaja dan bisa dipertanggungjawabkan langkah demi langkahnya.

**2. Apakah 490 baris "lebih benar" dibanding 515 baris?**

Ya, 490 baris lebih *veracious* (lebih bisa dipercaya), bukan berarti 515 baris salah secara mutlak. Konsep **Veracity** menekankan bahwa nilai suatu data tidak ditentukan oleh *Volume*-nya (banyaknya baris), melainkan oleh seberapa akurat dan layak dipercaya data tersebut untuk dijadikan dasar analisis atau pengambilan keputusan. 515 baris masih mengandung transaksi yang tercatat dua kali, yang jika dihitung akan menggandakan nilai penjualan secara keliru, serta baris tanpa identitas pembeli atau metode pembayaran yang jelas. 490 baris adalah subset yang sudah melewati verifikasi kualitas, sehingga lebih representatif terhadap kondisi transaksi yang sebenarnya, meskipun jumlahnya lebih kecil.

**3. Bagaimana menjelaskan kolom `rating` yang dibiarkan kosong ke tim Finance?**

Kepada tim Finance dapat dijelaskan bahwa `rating` memang bersifat opsional — pembeli tidak diwajibkan memberi rating setelah bertransaksi, sehingga nilai kosong (`NaN`) pada kolom ini adalah kondisi yang valid dan wajar, bukan data yang rusak. Jika nilai kosong ini dipaksa diisi dengan angka tebakan, maka "rata-rata rating" yang dihasilkan justru menjadi bias dan tidak mencerminkan preferensi pembeli yang sebenarnya. Solusi yang benar adalah menghitung rata-rata rating hanya dari baris yang benar-benar memiliki nilai rating, lalu melaporkan juga persentase transaksi yang memiliki rating agar konteksnya jelas, misalnya "rata-rata rating 4.2 dari 66% total transaksi yang memberi rating".

## Pertanyaan Evaluasi

**1. Perbedaan `dropna()` vs `fillna()`, kapan masing-masing dipakai?**

`dropna()` membuang seluruh baris yang memiliki nilai kosong pada kolom tertentu, sehingga baris tersebut hilang dari dataset. Ini dipakai ketika kolom bersifat wajib/esensial, seperti `customer_name` dan `payment_method` pada praktikum ini. `fillna()` menggantikan nilai kosong dengan nilai pengganti tanpa menghapus barisnya, sehingga jumlah baris tetap utuh. Ini dipakai ketika baris masih berguna untuk analisis lain meski satu kolomnya kosong dan tersedia nilai pengganti yang masuk akal, seperti `shipping_city` yang diisi `"Tidak Diketahui"`.

**2. Mengapa `drop_duplicates()` sebaiknya dijalankan di awal, bukan di akhir?**

Karena baris *duplicate* yang belum dibersihkan akan ikut mempengaruhi seluruh proses selanjutnya jika dibiarkan di akhir — misalnya ikut mempengaruhi perhitungan *missing value* per kolom atau ikut terhitung dua kali dalam analisis. Dengan menghapus *duplicate* sejak awal, seluruh langkah berikutnya bekerja di atas data yang jumlah barisnya sudah akurat.

**3. Risiko `pd.to_datetime(..., format="mixed", dayfirst=True)` pada data campuran ISO dan non-ISO?**

Risikonya adalah salah tafsir pada tanggal yang sebenarnya sudah tidak ambigu. `dayfirst=True` memberi instruksi untuk mengutamakan format "hari duluan" pada tanggal yang ambigu, namun tanggal berformat ISO (`YYYY-MM-DD`) yang sudah jelas tetap bisa ikut "dibalik", sehingga `2026-07-11` (11 Juli) salah terbaca menjadi 7 November. Solusi amannya adalah mencoba format eksplisit satu per satu untuk tiap nilai, seperti fungsi `parse_tanggal()` di Langkah K-5.

**4. Mengapa `rating` tidak di-*impute*, sedangkan `shipping_city` diisi "Tidak Diketahui"?**

`shipping_city` adalah data kategorikal deskriptif — mengisinya dengan label netral tidak mengubah makna data lain. Sebaliknya, `rating` adalah penilaian subjektif pembeli; nilai kosongnya berarti "pembeli tidak memberi rating", bukan "tidak diketahui". Jika diisi dengan angka tebakan, nilai buatan itu akan ikut dihitung seolah-olah penilaian sungguhan, sehingga mendistorsi analisis rating ke depannya.

**5. Keterkaitan praktikum ini dengan dimensi Veracity pada 5V?**

Seluruh proses di praktikum ini — deteksi *missing value*, penghapusan *duplicate*, koreksi tipe data, dan standardisasi format — pada dasarnya adalah upaya meningkatkan **Veracity** data, yaitu tingkat keakuratan dan kelayakan-dipercayaan data untuk dijadikan dasar analisis. Data mentah memiliki *Volume* yang sama, tetapi *Veracity*-nya rendah karena mengandung *noise*. Melalui *preprocessing*, praktikum ini menerapkan prinsip bahwa *Volume* besar tidak bernilai apa-apa jika *Veracity*-nya buruk ("*garbage in, garbage out*").